# 2. QLoRA demo: konfiguracja, trening i reload
Notebook domyślnie tylko czyta zamrożone artefakty. Trening wymaga świadomego ustawienia `RUN_TRAINING=True`; protected splits pozostają zamknięte.

In [ ]:
from pathlib import Path
import json, subprocess, sys

def project_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'pyproject.toml').exists(): return candidate
    raise RuntimeError('Nie znaleziono repozytorium')

ROOT = project_root()
load = lambda relative: json.loads((ROOT / relative).read_text(encoding='utf-8'))
config = load('configs/qlora_demo_v1.json')
config['id'], config['model']['id'], config['lora'], config['training']['max_steps']

In [ ]:
RUN_TRAINING = False
command = [sys.executable, '-m', 'peft_workshop.train', '--config', 'configs/qlora_demo_v1.json']
print('Polecenie:', ' '.join(command))
if RUN_TRAINING:
    subprocess.run(command, cwd=ROOT, check=True)
else:
    print('Trening pominięty. Ustaw RUN_TRAINING=True dopiero po sprawdzeniu GPU i cache modelu.')

## Zweryfikowany wynik 12 kroków
Te liczby pokazują pipeline, koszt i artefakt. Nie są dowodem jakości biznesowej.

In [ ]:
train = load('results/sprint3/q1_demo_training_metrics.json')
manifest = load('results/sprint3/q1_demo_adapter_manifest.json')
reload = load('results/sprint3/q1_demo_reload_smoke_metrics.json')
weights = next(item['bytes'] for item in manifest['files'] if item['name'] == 'adapter_model.safetensors')
summary = {
    'status': train['status'],
    'steps': config['training']['max_steps'],
    'wall_clock_s': train['wall_clock_seconds'],
    'peak_gpu_allocated_gib': train['peak_gpu_allocated_gib'],
    'truncated_cases': train['token_stats']['truncated_case_count'],
    'trainable_percent': train['model']['trainable_percent'],
    'adapter_weights_mb': round(weights / 1_000_000, 1),
    'reload_schema_valid': reload['aggregate']['schema_valid_rate'],
    'reload_sources_valid': reload['aggregate']['sources_valid_rate'],
    'reload_max_new_tokens': reload['metadata']['max_new_tokens'],
}
summary

In [ ]:
# Krótki wykres tekstowy lossu — bez dodatkowych bibliotek.
history = [row for row in train['log_history'] if 'loss' in row]
for row in history:
    bar = '█' * max(1, round(row['loss'] * 20))
    print(f"step {row['step']:>2}: {row['loss']:.3f} {bar}")

## Lekcja operacyjna: 128 vs 384 tokeny
Pierwsza próba reloadu przy 128 tokenach dała poprawny JSON, lecz niepełny schemat. Kontrolowany rerun przy 384 tokenach przeszedł wszystkie kontrole 1/1. Limit generacji jest częścią kontraktu testu.

## Pytanie do grupy
Które z liczb powyżej mówią o mechanice pipeline, a które rzeczywiście mogłyby wspierać decyzję o jakości?